In [1]:
from pathlib import Path
import deepecohab

#### Make project

Uses example data provided in the repository. Currently only default ecohab layout, more examples soon...

In [2]:
example_dir = Path.cwd()

In [3]:
config_path = deepecohab.create_ecohab_project(
	project_location=example_dir,  # Location where the project will be created
	experiment_name="test",  # Name of the project
	data_path=example_dir / "test_data",  # Folder with ecohab txt files
	light_phase_start="00:00:00",
	dark_phase_start="12:00:00",
	animal_ids=[
		"75DFFF1904",
		"352DE61A04",
		"9A44E61A04",
		"0627A81904",
		"A4939C1A04",
		"7376E61A04",
		"956BA81904",
		"325CE61A04",
		"85EDFF1904",
	],
)

Project already exists! Loading: /home/ula/DeepEcoHab/examples/test_2026-02-11/config.toml


#### Make structure

In [4]:
lf = deepecohab.get_ecohab_data_structure(config_path, fname_prefix="20", sanitize_animal_ids=True)

In [5]:
lf.collect()

antenna,time_under,animal_id,datetime,phase,day,hour,phase_count,time_spent,position
i8,i64,enum,"datetime[μs, Europe/Warsaw]",enum,i16,i8,u32,f64,cat
4,359,"""9A44E61A04""",2023-05-24 12:59:09.531 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
3,205,"""9A44E61A04""",2023-05-24 12:59:10.758 CEST,"""dark_phase""",1,12,1,1.23,"""c3_c2"""
2,153,"""A4939C1A04""",2023-05-24 12:59:14.314 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
1,307,"""A4939C1A04""",2023-05-24 12:59:16.367 CEST,"""dark_phase""",1,12,1,2.05,"""c2_c1"""
2,255,"""352DE61A04""",2023-05-24 12:59:16.972 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
…,…,…,…,…,…,…,…,…,…
3,50,"""A4939C1A04""",2023-05-29 23:52:10.486 CEST,"""dark_phase""",6,23,6,22.57,"""cage_2"""
3,50,"""85EDFF1904""",2023-05-29 23:52:10.635 CEST,"""dark_phase""",6,23,6,105.38,"""cage_2"""
4,50,"""85EDFF1904""",2023-05-29 23:52:10.839 CEST,"""dark_phase""",6,23,6,0.2,"""c2_c3"""


#### Chasings and ranking

In [6]:
chasings = deepecohab.calculate_chasings(config_path)
ranking = deepecohab.calculate_ranking(config_path)

In [7]:
from deepecohab.utils import auxfun
from deepecohab.core import create_data_structure
import polars as pl

cfg =auxfun.read_config(config_path)
antenna_pairs = cfg["antenna_combinations"]
antenna_pairs
cages: list[str] = cfg["cages"]
tunnels: list[str] = cfg["tunnels"]
#auxfun.remove_tunnel_directionality(lf, cfg)

In [8]:
from deepecohab.antenna_analysis.chasings import calculate_tube_tests

calculate_tube_tests(cfg).collect().select('tube_tests').sum()

tube_tests
u32
241


In [ ]:
calculate_tube_tests(cfg, 'CHASE', overwrite=True).collect().select('tube_tests').sum()

tube_tests
u32
241


In [ ]:
calculate_tube_tests(cfg, 'GUARD', overwrite=True).collect().select('tube_tests').sum()

tube_tests
u32
241


In [41]:
tunnel_dict = {
    "1": 'c1_c2',
    "2": 'c2_c1',
    "3": 'c2_c3',
    "4": 'c3_c2',
    "5": 'c3_c4',
    "6": 'c4_c3',
    "7": 'c4_c1',
    "8": 'c1_c4',
}


In [42]:
lf.collect()

antenna,time_under,animal_id,datetime,phase,day,hour,phase_count,time_spent,position
i8,i64,enum,"datetime[μs, Europe/Warsaw]",enum,i16,i8,u32,f64,cat
4,359,"""9A44E61A04""",2023-05-24 12:59:09.531 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
3,205,"""9A44E61A04""",2023-05-24 12:59:10.758 CEST,"""dark_phase""",1,12,1,1.23,"""c3_c2"""
2,153,"""A4939C1A04""",2023-05-24 12:59:14.314 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
1,307,"""A4939C1A04""",2023-05-24 12:59:16.367 CEST,"""dark_phase""",1,12,1,2.05,"""c2_c1"""
2,255,"""352DE61A04""",2023-05-24 12:59:16.972 CEST,"""dark_phase""",1,12,1,0.0,"""undefined"""
…,…,…,…,…,…,…,…,…,…
3,50,"""A4939C1A04""",2023-05-29 23:52:10.486 CEST,"""dark_phase""",6,23,6,22.57,"""cage_2"""
3,50,"""85EDFF1904""",2023-05-29 23:52:10.635 CEST,"""dark_phase""",6,23,6,105.38,"""cage_2"""
4,50,"""85EDFF1904""",2023-05-29 23:52:10.839 CEST,"""dark_phase""",6,23,6,0.2,"""c2_c3"""


In [ ]:
def update_position(lf: pl.LazyFrame):
    lf = lf.with_columns(
            pl.struct("position", "antenna").rle_id().over('animal_id').alias('run_id')
        ).with_columns(
            pl.cum_count("position").over(
                ["animal_id",'run_id']
            ).alias('consecutive_antenna_readout')
        )
    lf = lf.with_columns(
        pl.when(
            pl.col('consecutive_antenna_readout').mod(2) == 0
        ).then(
            pl.col("antenna").cast(pl.Utf8).replace(tunnel_dict).cast(pl.Categorical)
        ).otherwise(
            pl.col('position')
        ).alias('position')
    )

    return lf

def update_position2(lf: pl.LazyFrame):
    lf = lf.with_columns(
            pl.struct("position", "antenna").rle_id().over('animal_id').alias('run_id')
        ).with_columns(
            pl.cum_count("position").over(
                ["animal_id",'run_id']
            ).alias('consecutive_antenna_readout')
        )
    lf = lf.with_columns(
        pl.when(
            (pl.col('consecutive_antenna_readout') > 1) #& (pl.col('consecutive_antenna_readout') < pl.col('consecutive_antenna_readout').max().over(['animal_id', "run_id"]))
        ).then(
            pl.col("antenna").cast(pl.Utf8).replace(tunnel_dict).cast(pl.Categorical)
        ).otherwise(
            pl.col('position')
        ).alias('position')
    )

    return lf

In [ ]:
def calculate_tube_test(lf):
    lf1 = auxfun.remove_tunnel_directionality(lf, cfg)
    loser = lf1.with_columns(
            (pl.col('datetime') - pl.duration(seconds=pl.col("time_spent"))).alias("tunnel_entry"),
            pl.col("position").shift(1).over("animal_id").alias("prev_position")
        ).filter(
            ~pl.col("position").is_in(cages),
            (pl.col("prev_position") == pl.col("position").shift(-1).over("animal_id"))
        )

    winner = lf1.with_columns(
        (pl.col('datetime') - pl.duration(seconds=pl.col("time_spent"))).alias("tunnel_entry"),
        pl.col("position").shift(1).over("animal_id").alias("prev_position")
    )

    intermediate = loser.join(
        winner, on=["phase", "day", "phase_count", "position"], suffix="_winner" #sprawdzic bez hour
    ).filter(
        pl.col("animal_id") != pl.col("animal_id_winner"),
        pl.col("prev_position") != pl.col("prev_position_winner"),
        pl.col("datetime") < pl.col("datetime_winner"),
    ).with_columns(
        (
            pl.min_horizontal(["datetime", "datetime_winner"])
            - pl.max_horizontal(["tunnel_entry", "tunnel_entry_winner"])
        )
        .dt.total_seconds(fractional=True)
        .alias("overlap_duration")
    ).filter(pl.col("overlap_duration") > 0)
    return intermediate

In [67]:
#original
lf_or = update_position(lf)
int_or = calculate_tube_test(lf_or)

significant_cols = ['animal_id',
 'run_id',
 'position',
 'datetime',
 'consecutive_antenna_readout',
 'phase',
 'day',
 'phase_count',
 'hour',
 'time_spent',
 'tunnel_entry',
 'prev_position',
 'animal_id_winner',
 'run_id_winner',
 'datetime_winner',
 'consecutive_antenna_readout_winner',
 'hour_winner',
 'time_spent_winner',
 'tunnel_entry_winner',
 'prev_position_winner',
 'overlap_duration']
int_or.collect().to_pandas()

,antenna,time_under,animal_id,datetime,phase,day,hour,phase_count,time_spent,position,...,time_under_winner,animal_id_winner,datetime_winner,hour_winner,time_spent_winner,run_id_winner,consecutive_antenna_readout_winner,tunnel_entry_winner,prev_position_winner,overlap_duration
0,7,405,352DE61A04,2023-05-24 12:59:26.750000+02:00,dark_phase,1,12,1,0.61,tunnel_4,...,351,325CE61A04,2023-05-24 12:59:27.679000+02:00,12,2.23,1,1,2023-05-24 12:59:25.449000+02:00,undefined,0.610
1,5,102,85EDFF1904,2023-05-24 12:59:45.729000+02:00,dark_phase,1,12,1,3.58,tunnel_3,...,1181,0627A81904,2023-05-24 12:59:47.703000+02:00,12,3.78,2,1,2023-05-24 12:59:43.923000+02:00,tunnel_3,1.806
2,2,257,325CE61A04,2023-05-24 13:00:08.756000+02:00,dark_phase,1,13,1,0.36,tunnel_1,...,414,9A44E61A04,2023-05-24 13:00:11.465000+02:00,13,4.78,9,1,2023-05-24 13:00:06.685000+02:00,cage_1,0.360
3,6,924,85EDFF1904,2023-05-24 13:00:22.876000+02:00,dark_phase,1,13,1,6.36,tunnel_3,...,401,352DE61A04,2023-05-24 13:00:23.727000+02:00,13,4.75,9,1,2023-05-24 13:00:18.977000+02:00,cage_3,3.899
4,6,924,85EDFF1904,2023-05-24 13:00:22.876000+02:00,dark_phase,1,13,1,6.36,tunnel_3,...,311,9A44E61A04,2023-05-24 13:00:24.873000+02:00,13,4.00,13,1,2023-05-24 13:00:20.873000+02:00,cage_3,2.003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,5,50,9A44E61A04,2023-05-29 19:02:49.809000+02:00,dark_phase,6,19,6,23.51,tunnel_3,...,50,956BA81904,2023-05-29 19:02:50.006000+02:00,19,0.49,4197,1,2023-05-29 19:02:49.516000+02:00,cage_3,0.293
237,5,50,9A44E61A04,2023-05-29 19:02:49.809000+02:00,dark_phase,6,19,6,23.51,tunnel_3,...,153,7376E61A04,2023-05-29 19:04:08.204000+02:00,19,1339.14,11064,1,2023-05-29 18:41:49.064000+02:00,cage_4,23.510
238,4,101,0627A81904,2023-05-29 19:19:59.699000+02:00,dark_phase,6,19,6,1.00,tunnel_2,...,101,85EDFF1904,2023-05-29 19:19:59.837000+02:00,19,0.39,6801,1,2023-05-29 19:19:59.447000+02:00,cage_2,0.252
239,3,101,0627A81904,2023-05-29 19:44:25.059000+02:00,dark_phase,6,19,6,1.17,tunnel_2,...,104,7376E61A04,2023-05-29 19:44:25.159000+02:00,19,0.68,11176,1,2023-05-29 19:44:24.479000+02:00,tunnel_2,0.580


In [59]:
lfg = update_position2(lf)
lfg = lfg.group_by(
    ['animal_id', 'run_id']
).agg(
    pl.col('position').top_k(2).last(),
    pl.col('datetime').last().alias('datetime'),
    pl.col('consecutive_antenna_readout').last().alias('consecutive_antenna_readout'),
    pl.col('phase').last(),
    pl.col('day').last(),
    pl.col('phase_count').last(),
    pl.col('hour').last()
).sort('datetime')

lfg = create_data_structure.calculate_time_spent(lfg)

int_grp = calculate_tube_test(lfg)
int_grp.collect().to_pandas()

,animal_id,run_id,position,datetime,consecutive_antenna_readout,phase,day,phase_count,hour,time_spent,...,prev_position,animal_id_winner,run_id_winner,datetime_winner,consecutive_antenna_readout_winner,hour_winner,time_spent_winner,tunnel_entry_winner,prev_position_winner,overlap_duration
0,956BA81904,10,tunnel_4,2023-05-24 13:09:14.410000+02:00,12,dark_phase,1,1,13,26.20,...,tunnel_4,352DE61A04,65,2023-05-24 13:09:24.039000+02:00,3,13,37.49,2023-05-24 13:08:46.549000+02:00,cage_4,26.200
1,A4939C1A04,149,tunnel_2,2023-05-24 13:55:03.385000+02:00,3,dark_phase,1,1,13,123.52,...,tunnel_2,352DE61A04,277,2023-05-24 13:55:03.590000+02:00,1,13,0.50,2023-05-24 13:55:03.090000+02:00,cage_2,0.295
2,325CE61A04,273,tunnel_4,2023-05-24 13:53:20.348000+02:00,1,dark_phase,1,1,13,6.32,...,tunnel_4,7376E61A04,187,2023-05-24 13:56:44.240000+02:00,3,13,314.90,2023-05-24 13:51:29.340000+02:00,tunnel_3,6.320
3,325CE61A04,274,tunnel_4,2023-05-24 13:53:25.821000+02:00,3,dark_phase,1,1,13,5.47,...,tunnel_4,7376E61A04,187,2023-05-24 13:56:44.240000+02:00,3,13,314.90,2023-05-24 13:51:29.340000+02:00,tunnel_3,5.470
4,75DFFF1904,87,tunnel_4,2023-05-24 13:53:41.169000+02:00,4,dark_phase,1,1,13,36.90,...,tunnel_4,7376E61A04,187,2023-05-24 13:56:44.240000+02:00,3,13,314.90,2023-05-24 13:51:29.340000+02:00,tunnel_3,36.900
5,7376E61A04,2655,tunnel_4,2023-05-25 00:39:33.951000+02:00,3,light_phase,2,1,0,24.67,...,tunnel_4,352DE61A04,1604,2023-05-25 00:39:34.210000+02:00,1,0,0.56,2023-05-25 00:39:33.650000+02:00,cage_4,0.301
6,7376E61A04,4144,tunnel_2,2023-05-25 17:26:01.892000+02:00,3,dark_phase,2,2,17,6.79,...,tunnel_2,0627A81904,2755,2023-05-25 17:56:11.038000+02:00,1,17,2060.89,2023-05-25 17:21:50.148001+02:00,cage_2,6.790
7,7376E61A04,6271,tunnel_3,2023-05-26 15:44:09.799000+02:00,1,dark_phase,3,3,15,13.63,...,cage_3,352DE61A04,4374,2023-05-26 15:44:09.834000+02:00,1,15,0.24,2023-05-26 15:44:09.594000+02:00,cage_4,0.205
8,325CE61A04,2713,tunnel_3,2023-05-26 22:56:17.540000+02:00,3,dark_phase,3,3,22,30.25,...,tunnel_3,7376E61A04,6827,2023-05-26 22:56:18.278000+02:00,1,22,1.66,2023-05-26 22:56:16.618000+02:00,cage_3,0.922
9,A4939C1A04,2202,tunnel_3,2023-05-27 00:47:06+02:00,3,light_phase,4,3,0,18.65,...,tunnel_3,0627A81904,5909,2023-05-27 00:47:06.088000+02:00,1,0,0.44,2023-05-27 00:47:05.648000+02:00,cage_4,0.352


In [47]:
lf_or.collect().shape

(59834, 12)

In [52]:
lf_or.filter(pl.col('consecutive_antenna_readout')==1).collect().shape

(58916, 12)

In [56]:
int_grp.collect().columns

['animal_id',
 'run_id',
 'position',
 'datetime',
 'consecutive_antenna_readout',
 'phase',
 'day',
 'phase_count',
 'hour',
 'time_spent',
 'tunnel_entry',
 'prev_position',
 'animal_id_winner',
 'run_id_winner',
 'datetime_winner',
 'consecutive_antenna_readout_winner',
 'hour_winner',
 'time_spent_winner',
 'tunnel_entry_winner',
 'prev_position_winner',
 'overlap_duration']

In [ ]:
from itertools import product

def check_activity_for_runs(lf):
    lfg = lf.with_columns(
            pl.struct("position", "antenna").rle_id().over('animal_id').alias('run_id')
        )
    
    lfg = lfg.filter(
        pl.len().over("run_id", 'animal_id') > 1
    )

    lfg = lfg.with_columns(
        pl.col("antenna").cast(pl.Utf8).replace(tunnel_dict).cast(pl.Categorical).alias('position')
    ).group_by(
        "position", 'animal_id', 'run_id'
    ).agg(
        pl.col('datetime').last().alias('datetime'),
        pl.col('antenna').last().alias('antenna'),
        pl.col('time_spent').sum().alias('time_spent'),
        pl.col('phase').last(),
        pl.col('day').last(),
        pl.col('phase_count').last(),
        pl.col('hour').last(),
        pl.col('position').first().alias('tunnel_orig')
    )


    #lfg = auxfun.remove_tunnel_directionality(lfg, cfg)
    #lf = auxfun.remove_tunnel_directionality(lf, cfg)
    
    lfg = lfg.with_columns(
            (pl.col('datetime') - pl.duration(seconds=pl.col("time_spent"))).alias("tunnel_entry")
        )

    print(lfg.collect().shape)

    lf = lf.with_columns(
            (pl.col('datetime') - pl.duration(seconds=pl.col("time_spent"))).alias("tunnel_entry"),
            pl.col('position').first().alias('tunnel_orig')
        )
    



    intermediate = lfg.join(
        lf, on=["phase", "day", "phase_count", "position"], suffix="_winner"
    ).filter(
        pl.col("animal_id") != pl.col("animal_id_winner"),
        pl.col("tunnel_orig") != pl.col("tunnel_orig_winner"),
    ).with_columns(
        (
            pl.min_horizontal(["datetime", "datetime_winner"])
            - pl.max_horizontal(["tunnel_entry", "tunnel_entry_winner"])
        )
        .dt.total_seconds(fractional=True)
        .alias("overlap_duration")
    ).filter(pl.col("overlap_duration") > 0)
    return intermediate
    


In [106]:
check_activity_for_runs(lf).collect()

(647, 12)


position,animal_id,run_id,datetime,antenna,time_spent,phase,day,phase_count,hour,tunnel_orig,tunnel_entry,antenna_winner,time_under,animal_id_winner,datetime_winner,hour_winner,time_spent_winner,tunnel_entry_winner,tunnel_orig_winner,overlap_duration
cat,enum,u32,"datetime[μs, Europe/Warsaw]",i8,f64,enum,i16,u32,i8,cat,"datetime[μs, Europe/Warsaw]",i8,i64,enum,"datetime[μs, Europe/Warsaw]",i8,f64,"datetime[μs, Europe/Warsaw]",cat,f64


In [32]:
tube_tests.collect().to_pandas()

,phase,day,phase_count,winner,loser,hour,tube_tests
0,dark_phase,4,4,0627A81904,325CE61A04,20,1
1,dark_phase,4,4,0627A81904,352DE61A04,16,1
2,dark_phase,4,4,0627A81904,7376E61A04,0,0
3,dark_phase,4,4,0627A81904,75DFFF1904,0,0
4,dark_phase,4,4,0627A81904,85EDFF1904,0,0
...,...,...,...,...,...,...,...
797,dark_phase,5,5,A4939C1A04,7376E61A04,0,0
798,dark_phase,5,5,A4939C1A04,75DFFF1904,0,0
799,dark_phase,5,5,A4939C1A04,85EDFF1904,0,0
800,dark_phase,5,5,A4939C1A04,956BA81904,17,1


In [27]:
lf = update_position(lf)
lf.collect().filter(pl.col('animal_id')=="9A44E61A04").select(['antenna', 'consecutive_antenna_readout', 'position', 'time_spent']).to_pandas()

,antenna,consecutive_antenna_readout,position,time_spent
0,4,1,undefined,0.00
1,3,1,c3_c2,1.23
2,2,1,cage_2,9.91
3,1,1,c2_c1,1.08
4,8,1,cage_1,7.51
...,...,...,...,...
5346,6,1,c3_c4,3.08
5347,6,1,cage_4,90.49
5348,5,1,c4_c3,0.39
5349,5,1,cage_3,1808.83


In [15]:
chasings.collect()#.select('chasings').sum()

phase,day,phase_count,chaser,chased,hour,chasings
enum,i16,u32,enum,enum,i8,u32
"""dark_phase""",1,1,"""325CE61A04""","""7376E61A04""",18,1
"""dark_phase""",1,1,"""85EDFF1904""","""7376E61A04""",23,3
"""light_phase""",2,1,"""9A44E61A04""","""75DFFF1904""",0,1
"""dark_phase""",3,3,"""75DFFF1904""","""9A44E61A04""",13,1
"""dark_phase""",3,3,"""7376E61A04""","""75DFFF1904""",14,1
…,…,…,…,…,…,…
"""light_phase""",5,4,"""A4939C1A04""","""352DE61A04""",0,0
"""light_phase""",3,2,"""A4939C1A04""","""7376E61A04""",0,0
"""dark_phase""",3,3,"""A4939C1A04""","""85EDFF1904""",0,0


#### Activity

In [8]:
activity = deepecohab.calculate_activity(config_path)
cage_occupancy = deepecohab.calculate_cage_occupancy(config_path)

In [9]:
activity.schema

/tmp/ipykernel_6678/2129096252.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  activity.schema


Schema([('phase', Enum(categories=['light_phase', 'dark_phase'])),
        ('day', Int16),
        ('phase_count', UInt32),
        ('position', Categorical),
        ('animal_id',
         Enum(categories=['0627A81904', '325CE61A04', '352DE61A04', '7376E61A04', '75DFFF1904', '85EDFF1904', '956BA81904', '9A44E61A04', 'A4939C1A04'])),
        ('time_in_position', Float64),
        ('visits_to_position', UInt32)])

In [10]:
cage_occupancy.schema

/tmp/ipykernel_6678/2114962814.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  cage_occupancy.schema


Schema([('hour', Int8),
        ('day', Int16),
        ('animal_id',
         Enum(categories=['0627A81904', '325CE61A04', '352DE61A04', '7376E61A04', '75DFFF1904', '85EDFF1904', '956BA81904', '9A44E61A04', 'A4939C1A04'])),
        ('cage', Categorical),
        ('time_spent', UInt32)])

#### Sociability

In [11]:
pairwise_meetings = deepecohab.calculate_pairwise_meetings(config_path, minimum_time=2)
time_alone = deepecohab.calculate_time_alone(config_path)
in_cohort_sociability = deepecohab.calculate_incohort_sociability(config_path)

In [12]:
in_cohort_sociability.schema

/tmp/ipykernel_6678/3226582910.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  in_cohort_sociability.schema


Schema([('phase', Enum(categories=['light_phase', 'dark_phase'])),
        ('day', Int16),
        ('phase_count', UInt32),
        ('animal_id',
         Enum(categories=['0627A81904', '325CE61A04', '352DE61A04', '7376E61A04', '75DFFF1904', '85EDFF1904', '956BA81904', '9A44E61A04', 'A4939C1A04'])),
        ('animal_id_2',
         Enum(categories=['0627A81904', '325CE61A04', '352DE61A04', '7376E61A04', '75DFFF1904', '85EDFF1904', '956BA81904', '9A44E61A04', 'A4939C1A04'])),
        ('proportion_together', Float64),
        ('sociability', Float64)])

In [13]:
time_alone.schema

/tmp/ipykernel_6678/257197586.py:1: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  time_alone.schema


Schema([('day', Int16),
        ('phase', Enum(categories=['light_phase', 'dark_phase'])),
        ('animal_id',
         Enum(categories=['0627A81904', '325CE61A04', '352DE61A04', '7376E61A04', '75DFFF1904', '85EDFF1904', '956BA81904', '9A44E61A04', 'A4939C1A04'])),
        ('cage', Categorical),
        ('time_alone', UInt32),
        ('phase_count', UInt32)])

#### Dahsboard

In [14]:
deepecohab.run_dashboard(config_path)

In [15]:
import importlib
importlib.util.find_spec("deepecohab.dash.dashboard")


ModuleSpec(name='deepecohab.dash.dashboard', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f7ace8048f0>, origin='/home/ula/DeepEcoHab/deepecohab/dash/dashboard.py')

In [16]:
print(deepecohab.__file__)

/home/ula/DeepEcoHab/deepecohab/__init__.py
